In [1]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier

base = Path("/app")
dataset_dir = base / "ML_Training_datasets" / "Datasets" / "5S"
model_dir = base / "ML_Training_datasets" / "Models" / "5S_h10_long_only"
model_dir.mkdir(parents=True, exist_ok=True)

random_state = 42
target_col = "future_return_pct_10"
up_probability_thresholds = [0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80]


In [2]:
csv_files = sorted(dataset_dir.glob("memecoin_training_dataset_*.csv"))
dfs = []
for path in csv_files:
    df_part = pd.read_csv(path)
    df_part["source_file"] = path.name
    if "token_id" not in df_part.columns:
        token_stub = path.stem.replace("memecoin_training_dataset_", "token_")
        df_part["token_id"] = token_stub
    if "token_name" not in df_part.columns:
        df_part["token_name"] = df_part["token_id"]
    dfs.append(df_part)

if not dfs:
    raise ValueError(f"No dataset CSVs found in {dataset_dir}")

df = pd.concat(dfs, ignore_index=True)
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
df = df.sort_values(["token_id", "timestamp"]).reset_index(drop=True)

le_supertrend = LabelEncoder()
df["supertrend_trend_enc"] = le_supertrend.fit_transform(df["supertrend_trend"].astype(str))

def add_group_features(g):
    g = g.copy()
    g["rsi_lag_1"] = g["rsi"].shift(1)
    g["rsi_roll_3"] = g["rsi"].rolling(3, min_periods=1).mean()
    g["rsi_roll_5"] = g["rsi"].rolling(5, min_periods=1).mean()
    g["atr_roll_3"] = g["atr"].rolling(3, min_periods=1).mean()
    g["atr_roll_5"] = g["atr"].rolling(5, min_periods=1).mean()
    g["volume_avg_roll_3"] = g["volume_avg"].rolling(3, min_periods=1).mean()
    g["volume_avg_roll_5"] = g["volume_avg"].rolling(5, min_periods=1).mean()
    g["obv_roll_3"] = g["obv"].rolling(3, min_periods=1).mean()
    return g

token_id_backup = df["token_id"].copy()
df = df.groupby("token_id", group_keys=False).apply(add_group_features)
if "token_id" not in df.columns:
    df["token_id"] = token_id_backup.loc[df.index].to_numpy()

df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna().reset_index(drop=True)
print(df.shape)


(231014, 61)


In [3]:
base_feature_cols = [
    "rsi", "ema10", "ema20", "ema50", "ema100", "ema200", "ema_cross",
    "macd_line", "macd_signal", "macd_hist", "vwap", "adx", "plus_di",
    "minus_di", "stoch_k", "stoch_d", "boll_upper", "boll_middle",
    "boll_lower", "boll_percent", "atr", "obv", "supertrend_value",
    "cci", "roc", "momentum3", "volume_sum", "volume_avg", "range_pct",
    "open_rel", "high_rel", "low_rel", "close_rel", "hour", "minute",
    "weekday", "supertrend_trend_enc", "rsi_lag_1", "rsi_roll_3", "rsi_roll_5",
    "atr_roll_3", "atr_roll_5", "volume_avg_roll_3", "volume_avg_roll_5", "obv_roll_3"
]
optional_feature_cols = [
    "last_5m_buyCount", "last_5m_sellCount", "last_5m_buyVolumeSol",
    "last_5m_sellVolumeSol", "last_5m_priceSol"
]

feature_cols = base_feature_cols + [c for c in optional_feature_cols if c in df.columns]
X = df[feature_cols].copy()
tokens = np.array(sorted(df["token_id"].unique()))

train_tokens, test_tokens = train_test_split(tokens, test_size=0.2, random_state=random_state)
train_tokens, val_tokens = train_test_split(train_tokens, test_size=0.2, random_state=random_state)

train_mask = df["token_id"].isin(train_tokens)
val_mask = df["token_id"].isin(val_tokens)
test_mask = df["token_id"].isin(test_tokens)

X_train = X.loc[train_mask]
X_val = X.loc[val_mask]
X_test = X.loc[test_mask]

split_summary = pd.DataFrame([
    {"split": "train", "tokens": len(train_tokens), "rows": int(train_mask.sum())},
    {"split": "val", "tokens": len(val_tokens), "rows": int(val_mask.sum())},
    {"split": "test", "tokens": len(test_tokens), "rows": int(test_mask.sum())},
])
split_summary


,split,tokens,rows
0,train,12,159895
1,val,3,34947
2,test,4,36172


In [4]:
def direction_label(series):
    return pd.Series(np.where(series > 0.0, "up", "down_or_flat"), index=series.index)

y_train = direction_label(df.loc[train_mask, target_col])
y_val = direction_label(df.loc[val_mask, target_col])
y_test = direction_label(df.loc[test_mask, target_col])

dir_encoder = LabelEncoder()
y_train_enc = dir_encoder.fit_transform(y_train)
y_val_enc = dir_encoder.transform(y_val)
y_test_enc = dir_encoder.transform(y_test)

class_counts = y_train.value_counts().sort_index()
class_weight_scale = len(y_train) / (len(class_counts) * class_counts)
sample_weight = y_train.map(class_weight_scale).to_numpy()

clf_model = XGBClassifier(
    n_estimators=500,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=random_state,
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
)

clf_model.fit(
    X_train,
    y_train_enc,
    sample_weight=sample_weight,
    eval_set=[(X_val, y_val_enc)],
    verbose=False,
)

val_proba = clf_model.predict_proba(X_val)
test_proba = clf_model.predict_proba(X_test)
val_pred = clf_model.predict(X_val)
test_pred = clf_model.predict(X_test)

print("Validation accuracy:", accuracy_score(y_val_enc, val_pred))
print("Test accuracy:", accuracy_score(y_test_enc, test_pred))
print(classification_report(y_test_enc, test_pred, target_names=dir_encoder.classes_))


Validation accuracy: 0.5167539416831202
Test accuracy: 0.5225035939400642
              precision    recall  f1-score   support

down_or_flat       0.53      0.55      0.54     18452
          up       0.51      0.49      0.50     17720

    accuracy                           0.52     36172
   macro avg       0.52      0.52      0.52     36172
weighted avg       0.52      0.52      0.52     36172



In [5]:
up_class_index = int(np.where(dir_encoder.classes_ == "up")[0][0])

def evaluate_long_only_thresholds(y_true_enc, returns, proba, thresholds):
    up_prob = proba[:, up_class_index]
    rows = []

    for thr in thresholds:
        mask = up_prob >= thr
        selected = int(mask.sum())
        coverage = float(mask.mean())

        if selected == 0:
            rows.append({
                "threshold": thr,
                "selected_rows": 0,
                "coverage": coverage,
                "win_rate": np.nan,
                "avg_return_pct": np.nan,
                "median_return_pct": np.nan,
                "avg_positive_return_pct": np.nan,
                "avg_negative_return_pct": np.nan,
            })
            continue

        selected_returns = returns[mask]
        win_mask = selected_returns > 0
        loss_mask = selected_returns <= 0

        rows.append({
            "threshold": thr,
            "selected_rows": selected,
            "coverage": coverage,
            "win_rate": float(win_mask.mean()),
            "avg_return_pct": float(selected_returns.mean()),
            "median_return_pct": float(np.median(selected_returns)),
            "avg_positive_return_pct": float(selected_returns[win_mask].mean()) if win_mask.any() else np.nan,
            "avg_negative_return_pct": float(selected_returns[loss_mask].mean()) if loss_mask.any() else np.nan,
        })

    return pd.DataFrame(rows)

val_returns = df.loc[val_mask, target_col].to_numpy()
test_returns = df.loc[test_mask, target_col].to_numpy()

val_long_only_metrics = evaluate_long_only_thresholds(y_val_enc, val_returns, val_proba, up_probability_thresholds)
test_long_only_metrics = evaluate_long_only_thresholds(y_test_enc, test_returns, test_proba, up_probability_thresholds)

val_long_only_metrics


,threshold,selected_rows,coverage,win_rate,avg_return_pct,median_return_pct,avg_positive_return_pct,avg_negative_return_pct
0,0.50,16886,0.483189,0.512792,0.167451,0.057291,2.640003,-2.434935
1,0.55,11010,0.315049,0.519891,0.205918,0.080431,2.601774,-2.388458
2,0.60,6564,0.187827,0.526508,0.198311,0.110792,2.612950,-2.486692
3,0.65,3639,0.104129,0.530640,0.303998,0.122166,2.846348,-2.570287
4,0.70,1902,0.054425,0.557834,0.543231,0.272943,3.059924,-2.631811
5,0.75,859,0.024580,0.615832,1.099171,0.707106,3.438787,-2.651306
6,0.80,314,0.008985,0.605096,1.241192,0.718688,3.865670,-2.780186


In [6]:
test_long_only_metrics


,threshold,selected_rows,coverage,win_rate,avg_return_pct,median_return_pct,avg_positive_return_pct,avg_negative_return_pct
0,0.50,17010,0.470253,0.513169,0.514257,0.091161,4.329278,-3.507155
1,0.55,10883,0.300868,0.523936,0.769588,0.191836,5.179025,-4.083260
2,0.60,6686,0.184839,0.537840,1.124883,0.375115,6.355435,-4.962193
3,0.65,3834,0.105994,0.544079,1.451718,0.596268,7.718033,-6.026275
4,0.70,2162,0.059770,0.535615,1.325916,0.726887,8.551471,-7.007940
5,0.75,1132,0.031295,0.546820,1.550101,0.821626,9.003994,-7.443973
6,0.80,404,0.011169,0.579208,1.750257,1.792809,8.791113,-7.941274


In [7]:
best_row = test_long_only_metrics.sort_values(["avg_return_pct", "win_rate", "coverage"], ascending=[False, False, False]).iloc[0]
best_threshold = float(best_row["threshold"])
best_threshold


0.8

In [8]:
up_prob_test = test_proba[:, up_class_index]
best_mask = up_prob_test >= best_threshold
selected_test = df.loc[test_mask].reset_index(drop=True).loc[best_mask, ["token_id", "token_name", "timestamp", target_col]].copy()
selected_test["predicted_up_probability"] = up_prob_test[best_mask]

print(f"Best threshold: {best_threshold:.2f}")
print(f"Selected rows: {len(selected_test)} / {int(test_mask.sum())}")
selected_test.head(20)


Best threshold: 0.80
Selected rows: 404 / 36172


,token_id,token_name,timestamp,future_return_pct_10,predicted_up_probability
2219,2JBXEbEfzMNZGmP8fRhsUjJQqZUDLhJzwBjvh7vBw7Df,Jellybean,2026-02-28 06:31:35+00:00,3.203433,0.843322
2220,2JBXEbEfzMNZGmP8fRhsUjJQqZUDLhJzwBjvh7vBw7Df,Jellybean,2026-02-28 06:31:40+00:00,4.766730,0.887470
2221,2JBXEbEfzMNZGmP8fRhsUjJQqZUDLhJzwBjvh7vBw7Df,Jellybean,2026-02-28 06:31:45+00:00,4.244016,0.852706
2222,2JBXEbEfzMNZGmP8fRhsUjJQqZUDLhJzwBjvh7vBw7Df,Jellybean,2026-02-28 06:31:55+00:00,4.072538,0.829081
2223,2JBXEbEfzMNZGmP8fRhsUjJQqZUDLhJzwBjvh7vBw7Df,Jellybean,2026-02-28 06:32:05+00:00,3.643449,0.823022
2295,2JBXEbEfzMNZGmP8fRhsUjJQqZUDLhJzwBjvh7vBw7Df,Jellybean,2026-02-28 06:40:00+00:00,-8.652015,0.803148
2298,2JBXEbEfzMNZGmP8fRhsUjJQqZUDLhJzwBjvh7vBw7Df,Jellybean,2026-02-28 06:40:15+00:00,-3.384488,0.809519
2299,2JBXEbEfzMNZGmP8fRhsUjJQqZUDLhJzwBjvh7vBw7Df,Jellybean,2026-02-28 06:40:20+00:00,-3.546104,0.817705
2300,2JBXEbEfzMNZGmP8fRhsUjJQqZUDLhJzwBjvh7vBw7Df,Jellybean,2026-02-28 06:40:25+00:00,-1.523415,0.868675
2305,2JBXEbEfzMNZGmP8fRhsUjJQqZUDLhJzwBjvh7vBw7Df,Jellybean,2026-02-28 06:40:50+00:00,1.655554,0.896804


In [9]:
feature_importance = pd.Series(clf_model.feature_importances_, index=feature_cols).sort_values(ascending=False)

joblib.dump(clf_model, model_dir / "xgb_clf_h10_long_only.joblib")
joblib.dump({
    "feature_cols": feature_cols,
    "target_col": target_col,
    "up_probability_thresholds": up_probability_thresholds,
    "direction_classes": list(dir_encoder.classes_),
    "train_tokens": list(train_tokens),
    "val_tokens": list(val_tokens),
    "test_tokens": list(test_tokens),
}, model_dir / "artifacts_h10_long_only.joblib")

val_long_only_metrics.to_csv(model_dir / "val_long_only_metrics.csv", index=False)
test_long_only_metrics.to_csv(model_dir / "test_long_only_metrics.csv", index=False)
selected_test.to_csv(model_dir / "selected_test_signals.csv", index=False)
feature_importance.head(20)


momentum3               0.028288
ema100                  0.026741
ema50                   0.026724
supertrend_value        0.026724
boll_lower              0.026209
ema20                   0.025813
high_rel                0.025630
vwap                    0.025373
low_rel                 0.025353
ema200                  0.025243
volume_avg_roll_5       0.025149
ema10                   0.025146
boll_upper              0.025107
supertrend_trend_enc    0.024872
weekday                 0.024705
volume_avg_roll_3       0.024690
atr_roll_3              0.024018
boll_middle             0.023797
volume_sum              0.023415
hour                    0.023272
dtype: float32